# 6 · FAISS retrieval, persistence, and PDF Q&A

Use local BGE embeddings and an exact `IndexFlatIP` index with unit-normalized vectors. Inner-product search then gives cosine similarity. The New York brochure is a historical Margie's Travel training asset. Configure the root `.env` before running Azure cells. Requests may incur charges. Stored outputs are cleared; no new live Azure results are supplied.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT))
from lab.config import get_chat_model

In [ ]:
from lab.retrieval import FaissStore, load_pdf_chunks, get_embeddings, answer_question
documents = load_pdf_chunks(ROOT / "assets/new_york_brochure.pdf")
embeddings = get_embeddings()
store = FaissStore.build(documents, embeddings)
print("Chunks:", len(documents), "Embedding dimension:", store.index.d)

In [ ]:
question = "Which hotels are listed in the New York brochure?"
hits = store.search(question, k=3)
for hit in hits:
    print(hit.document.metadata, round(hit.score, 3))
print(answer_question(question, hits))

## Save and reload your own index
FAISS supports disk serialization. This helper saves `index.faiss` plus JSON document metadata, with no pickle deserialization. Load only files you created locally; the checksum detects accidental mismatch but does not establish trust.

In [ ]:
index_dir = ROOT / "generated/new_york_index"
store.save(index_dir)
reloaded = FaissStore.load(index_dir, embeddings)
reloaded_hits = reloaded.search(question, k=3)
print([(h.document.metadata, round(h.score, 3)) for h in reloaded_hits])

The original archive's prebuilt `index.faiss` and `index.pkl` are excluded. The Streamlit app rebuilds from the included PDF on its first question and caches the result in memory. Changing the PDF bytes or embedding model changes the app cache key. Generated indexes are ignored by Git.